# NYC Motor Vehicle Collisions Project

This notebook documents our exploratory analysis for the NYC Motor Vehicle Collisions Streamlit app.

Our goal is to use NYC Open Data to examine temporal and spatial patterns in motor vehicle collisions in 2026.

In [6]:
import pandas as pd
import altair as alt
from urllib.parse import urlencode
from datetime import datetime

In [7]:
def load_person_2026_live():
    base_url = "https://data.cityofnewyork.us/resource/f55k-p6yu.json"
    limit = 50000
    offset = 0
    all_data = []

    start_date = "2026-01-01T00:00:00"
    end_date = datetime.today().strftime("%Y-%m-%dT%H:%M:%S")

    while True:
        query_params = {
            "$where": f"crash_date between '{start_date}' and '{end_date}'",
            "$limit": limit,
            "$offset": offset,
        }

        url = f"{base_url}?{urlencode(query_params)}"
        df = pd.read_json(url)

        if df.empty:
            break

        all_data.append(df)
        offset += limit

    if all_data:
        return pd.concat(all_data, ignore_index=True)
    return pd.DataFrame()

person_df = load_person_2026_live()
person_df.head()

,unique_id,collision_id,crash_date,crash_time,person_id,person_type,person_injury,vehicle_id,ped_role,person_age,...,emotional_status,bodily_injury,position_in_vehicle,safety_equipment,complaint,person_sex,ped_location,ped_action,contributing_factor_1,contributing_factor_2
0,13580439,4869919,2026-01-01T00:00:00.000,2026-03-11 06:16:00,a877f299-5521-4965-8d38-9364cadf0e50,Occupant,Unspecified,20996199.0,Owner,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,13580440,4869919,2026-01-01T00:00:00.000,2026-03-11 06:16:00,01bc4f16-11e8-4b13-9952-2ce58fbf00e0,Occupant,Unspecified,NaN,Witness,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,13607952,4868890,2026-01-01T00:00:00.000,2026-03-11 22:12:00,b10b5429-5aaa-412f-8b2e-c54f03e1cefc,Occupant,Unspecified,21012017.0,Driver,40.0,...,Does Not Apply,Does Not Apply,Driver,None,Does Not Apply,F,NaN,NaN,NaN,NaN
3,13607464,4868944,2026-01-01T00:00:00.000,2026-03-11 17:48:00,8e01a740-2262-45f4-b147-8438c658871a,Occupant,Unspecified,21011713.0,Registrant,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,13606607,4869325,2026-01-01T00:00:00.000,2026-03-11 03:58:00,68360aa0-b5b1-4128-95ba-eebde4d62ca4,Occupant,Unspecified,21011220.0,Passenger,25.0,...,Does Not Apply,Does Not Apply,Unknown,Lap Belt & Harness,Does Not Apply,F,NaN,NaN,NaN,NaN


## Initial Exploration

We first explored the person-level dataset to understand its size, time range, and structure.

In [9]:
person_df.shape

(48712, 21)

In [8]:
person_df.columns

Index(['unique_id', 'collision_id', 'crash_date', 'crash_time', 'person_id',
       'person_type', 'person_injury', 'vehicle_id', 'ped_role', 'person_age',
       'ejection', 'emotional_status', 'bodily_injury', 'position_in_vehicle',
       'safety_equipment', 'complaint', 'person_sex', 'ped_location',
       'ped_action', 'contributing_factor_1', 'contributing_factor_2'],
      dtype='object')

In [10]:
person_df["crash_date"] = pd.to_datetime(person_df["crash_date"], errors="coerce")
person_df["crash_date"].min(), person_df["crash_date"].max()

(Timestamp('2026-01-01 00:00:00'), Timestamp('2026-03-07 00:00:00'))

## Crashes by Day of Week

We examined whether collisions are evenly distributed across the week.

In [11]:
unique_crashes = person_df.drop_duplicates(subset="collision_id").copy()
unique_crashes["weekday"] = unique_crashes["crash_date"].dt.day_name()

weekday_counts = unique_crashes["weekday"].value_counts().reset_index()
weekday_counts.columns = ["weekday", "crashes"]

weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

weekday_counts

,weekday,crashes
0,Friday,2518
1,Thursday,2358
2,Tuesday,2053
3,Saturday,2026
4,Wednesday,1967
5,Monday,1935
6,Sunday,1708


In [12]:
alt.Chart(weekday_counts).mark_bar().encode(
    x=alt.X("weekday:N", sort=weekday_order),
    y="crashes:Q",
    tooltip=["weekday", "crashes"]
)

alt.Chart(...)

### Observation
Crash counts are not evenly distributed across the week. Friday appears to have the highest number of crashes, while Sunday is lower, suggesting that weekday travel patterns may influence collision frequency.

In [15]:
def load_crash_2026_live():
    base_url = "https://data.cityofnewyork.us/resource/h9gi-nx95.json"
    limit = 50000
    offset = 0
    all_data = []

    start_date = "2026-01-01T00:00:00"
    end_date = datetime.today().strftime("%Y-%m-%dT%H:%M:%S")

    while True:
        query_params = {
            "$where": f"crash_date between '{start_date}' and '{end_date}'",
            "$limit": limit,
            "$offset": offset,
        }

        url = f"{base_url}?{urlencode(query_params)}"
        df = pd.read_json(url)

        if df.empty:
            break

        all_data.append(df)
        offset += limit

    if all_data:
        return pd.concat(all_data, ignore_index=True)

    return pd.DataFrame()

## Merging Datasets

We merged the person-level and crash-level datasets using `collision_id` to support broader analysis.

In [16]:
crash_df = load_crash_2026_live()
crash_df.head()

,crash_date,crash_time,borough,zip_code,latitude,longitude,location,cross_street_name,number_of_persons_injured,number_of_persons_killed,...,vehicle_type_code1,vehicle_type_code2,on_street_name,off_street_name,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5
0,2026-01-01T00:00:00.000,2026-03-11 18:15:00,MANHATTAN,10036.0,40.759254,-73.98691,"{'latitude': '40.759254', 'longitude': '-73.98...",250 W 46 ST,0,0,...,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-01-01T00:00:00.000,2026-03-11 15:30:00,MANHATTAN,10013.0,40.721058,-73.99921,"{'latitude': '40.721058', 'longitude': '-73.99...",39 CROSBY ST,0,0,...,Sedan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-01-01T00:00:00.000,2026-03-11 00:00:00,BROOKLYN,11212.0,40.657990,-73.90146,"{'latitude': '40.65799', 'longitude': '-73.901...",96 NEW LOTS AVE,0,0,...,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-01-01T00:00:00.000,2026-03-11 18:10:00,BRONX,10467.0,40.878746,-73.87254,"{'latitude': '40.878746', 'longitude': '-73.87...",NaN,0,0,...,Sedan,NaN,DECATUR AVE,E GUN HILL RD,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-01-01T00:00:00.000,2026-03-11 08:54:00,BROOKLYN,11207.0,40.651688,-73.88876,"{'latitude': '40.651688', 'longitude': '-73.88...",NaN,1,0,...,Station Wagon/Sport Utility Vehicle,Sedan,FLATLANDS AVE,ALABAMA AVE,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
crash_df["crash_date"] = pd.to_datetime(crash_df["crash_date"], errors="coerce")

merged_df = pd.merge(person_df, crash_df, on="collision_id", how="inner")

merged_df.shape

(48712, 49)

In [18]:
merged_df.head()

,unique_id,collision_id,crash_date_x,crash_time_x,person_id,person_type,person_injury,vehicle_id,ped_role,person_age,...,vehicle_type_code1,vehicle_type_code2,on_street_name,off_street_name,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5
0,13580439,4869919,2026-01-01,2026-03-11 06:16:00,a877f299-5521-4965-8d38-9364cadf0e50,Occupant,Unspecified,20996199.0,Owner,NaN,...,Station Wagon/Sport Utility Vehicle,NaN,OCEAN PKWY,AVENUE U,NaN,NaN,NaN,NaN,NaN,NaN
1,13580440,4869919,2026-01-01,2026-03-11 06:16:00,01bc4f16-11e8-4b13-9952-2ce58fbf00e0,Occupant,Unspecified,NaN,Witness,NaN,...,Station Wagon/Sport Utility Vehicle,NaN,OCEAN PKWY,AVENUE U,NaN,NaN,NaN,NaN,NaN,NaN
2,13607952,4868890,2026-01-01,2026-03-11 22:12:00,b10b5429-5aaa-412f-8b2e-c54f03e1cefc,Occupant,Unspecified,21012017.0,Driver,40.0,...,Sedan,NaN,E 89 ST,FOSTER AVE,NaN,NaN,NaN,NaN,NaN,NaN
3,13607464,4868944,2026-01-01,2026-03-11 17:48:00,8e01a740-2262-45f4-b147-8438c658871a,Occupant,Unspecified,21011713.0,Registrant,NaN,...,Bus,NaN,EASTERN PKWY,UTICA AVE,NaN,NaN,NaN,NaN,NaN,NaN
4,13606607,4869325,2026-01-01,2026-03-11 03:58:00,68360aa0-b5b1-4128-95ba-eebde4d62ca4,Occupant,Unspecified,21011220.0,Passenger,25.0,...,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,NEW ENGLAND THRUWAY,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Daily Trend Analysis

We examined how crash counts change over time in 2026.

In [ ]:
unique_merged = merged_df.drop_duplicates(subset="collision_id").copy()
unique_merged["date"] = unique_merged["crash_date_x"].dt.date

daily_counts = unique_merged.groupby("date").size().reset_index(name="crashes")
daily_counts.head()

In [ ]:
alt.Chart(daily_counts).mark_line().encode(
    x="date:T",
    y="crashes:Q",
    tooltip=["date", "crashes"]
)

### Observation
Daily crash counts fluctuate over time rather than following a strong upward or downward trend. Most days remain within a relatively stable range, suggesting that collisions are a persistent urban safety issue.

## Borough Analysis

We also compared crash counts across boroughs to explore spatial patterns.

In [ ]:
borough_counts = unique_merged["borough"].value_counts().reset_index()
borough_counts.columns = ["borough", "crashes"]
borough_counts

In [ ]:
alt.Chart(borough_counts).mark_bar().encode(
    x="borough:N",
    y="crashes:Q",
    tooltip=["borough", "crashes"]
)

### Observation
Crash counts differ across boroughs. Brooklyn has the highest number of crashes, followed by Queens, while Staten Island has the lowest. This suggests that collisions are spatially concentrated rather than evenly distributed across the city.

## Summary

This notebook documents the exploratory work that informed our Streamlit app. Our analysis suggests that NYC collision patterns vary across both time and location. The updated app builds on these findings through interactive visualizations and clearer explanations.